In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

from app.core.inference import model_loader
from app.core.config import settings

print(f"Корневая директория: {ROOT_DIR}")

Корневая директория: C:\Users\nicon\Desktop\pet


In [2]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: False


In [3]:
N_RUNS = 100
WARMUP_RUNS = 10
WINDOW_SIZE = 30

MODELS_TO_TEST = ["catboost", "rf", "lstm", "cnn"]
DATASETS_TO_TEST = ["FD001", "FD002", "FD003", "FD004"]

FEATURE_COUNTS = {
    "FD001": 17,
    "FD002": 24,
    "FD003": 19,
    "FD004": 24
}

In [4]:
def measure_inference_time(model_key: str, data: np.ndarray, n_runs: int = 100, warmup: int = 10) -> dict:
    """Измеряет время инференса модели."""
    for _ in range(warmup):
        model_loader.predict(model_key, data)
    
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        model_loader.predict(model_key, data)
        end = time.perf_counter()
        times.append((end - start) * 1000)
    
    times = np.array(times)
    
    return {
        "mean_ms": np.mean(times),
        "std_ms": np.std(times),
        "times": times
    }

def run_benchmark(datasets: list = None, models: list = None, n_runs: int = 100):
    """Запускает бенчмарк для всех моделей и датасетов."""
    if datasets is None:
        datasets = DATASETS_TO_TEST
    if models is None:
        models = MODELS_TO_TEST
    
    results = {}
    total_start = time.time()
    
    for dataset in datasets:
        print(f"Датасет: {dataset}")
        print("-" * 60)
        
        n_features = FEATURE_COUNTS[dataset]
        data = np.random.randn(WINDOW_SIZE, n_features).astype(np.float32)
        
        results[dataset] = {}
        
        for model_type in models:
            model_key = f"{model_type}_{dataset}"
            
            available_models = model_loader.get_available_models()
            if model_key not in available_models:
                print(f"{model_key} - не найдена")
                continue
            
            print(f"{model_type}:", end=" ")
            
            try:
                stats = measure_inference_time(model_key, data, n_runs=n_runs)
                results[dataset][model_type] = stats
                print(f"✅ {stats['mean_ms']:.2f} ± {stats['std_ms']:.2f} мс")
            except Exception as e:
                print(f"❌ Ошибка: {e}")
                results[dataset][model_type] = {"error": str(e)}

        print()
    
    total_time = time.time() - total_start
    
    return results

In [5]:
results = run_benchmark(n_runs=N_RUNS)

Датасет: FD001
------------------------------------------------------------
catboost: ✅ 1.18 ± 0.04 мс
rf: ✅ 25.22 ± 0.61 мс
lstm: ✅ 0.90 ± 0.08 мс
cnn: ✅ 0.53 ± 0.05 мс

Датасет: FD002
------------------------------------------------------------
catboost: ✅ 1.35 ± 0.05 мс
rf: ✅ 36.56 ± 0.87 мс
lstm: ✅ 0.95 ± 0.03 мс
cnn: ✅ 0.56 ± 0.02 мс

Датасет: FD003
------------------------------------------------------------
catboost: ✅ 1.23 ± 0.03 мс
rf: ✅ 47.08 ± 1.25 мс
lstm: ✅ 0.94 ± 0.07 мс
cnn: ✅ 0.57 ± 0.03 мс

Датасет: FD004
------------------------------------------------------------
catboost: ✅ 1.36 ± 0.05 мс
rf: ✅ 48.02 ± 2.05 мс
lstm: ✅ 0.91 ± 0.08 мс
cnn: ✅ 0.54 ± 0.06 мс



In [6]:
# Сводная таблица
rows = []
for dataset, models in results.items():
    row = {"Dataset": dataset}
    for model_type, stats in models.items():
        row[model_type] = f"{stats['mean_ms']:.2f} ± {stats['std_ms']:.2f}"
    rows.append(row)
    
summary_df = pd.DataFrame(rows)

summary_df

,Dataset,catboost,rf,lstm,cnn
0,FD001,1.18 ± 0.04,25.22 ± 0.61,0.90 ± 0.08,0.53 ± 0.05
1,FD002,1.35 ± 0.05,36.56 ± 0.87,0.95 ± 0.03,0.56 ± 0.02
2,FD003,1.23 ± 0.03,47.08 ± 1.25,0.94 ± 0.07,0.57 ± 0.03
3,FD004,1.36 ± 0.05,48.02 ± 2.05,0.91 ± 0.08,0.54 ± 0.06
